# Column cards

A column card summarizes one column using all rows. Counts use exact value identity; numeric summary
statistics use floating-point arithmetic over the full column; the role label is inferred from the dtype and
counts. Cards also report concentration measures and, for strings,
format signatures. This chapter describes how these fields are computed.


## Exact counting at any cardinality

Every count on a card — distinct, nulls, top values, concentration — is
exact. The counting routine is adaptive over a compact value key: signed and unsigned 64-bit
integers without a floating-point cast, floats as bit patterns (non-finite values treated as missing, −0.0 normalized
to +0.0), strings, and booleans. It accumulates in a hash map until the
map holds 4,096 distinct keys; if the column has more, it switches to
sorting the compact keys and counting runs. Both paths examine every row and
produce an exact distinct count; the
limit determines which counting algorithm runs.

Only the reported top values (top 64, ordered by count descending then key
ascending) own their string data; the high-cardinality sort borrows
strings. Concentration is computed over the full value distribution, not
over only the top-k values. `effective_cardinality` uses the counts of all
values, including on high-cardinality columns. The
implementation notes live in
[the counting chapter](../implementation/counting.ipynb).

## Role inference

Roles are assigned deterministically from dtype and exact counts:


In [1]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Role': ['`all_null`', '`constant`', '`boolean`, `date`, `datetime`', '`categorical` (strings)', '`free_text` (strings)', '`categorical` (numeric)', '`identifier` (numeric)', '`continuous` (numeric)', '`other`'], 'Rule': ['no non-null values', 'one distinct value', 'boolean / date / datetime dtype', '≤ 500 distinct values', 'more than 500 distinct values', '≤ 12 distinct with n ≥ 30, or distinct ratio ≤ 2% with ≤ 100 distinct and n ≥ 100', 'distinct ratio ≥ 0.98', 'otherwise', 'dtype the profiler treats as opaque (reported in coverage)']})
(GT(_doc_table)
 .fmt_markdown(columns=['Role', 'Rule'])
 .tab_header(title='Role inference')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


GT(_tbl_data=shape: (9, 2)
┌───────────────────────────────┬─────────────────────────────────┐
│ Role                          ┆ Rule                            │
│ ---                           ┆ ---                             │
│ str                           ┆ str                             │
╞═══════════════════════════════╪═════════════════════════════════╡
│ `all_null`                    ┆ no non-null values              │
│ `constant`                    ┆ one distinct value              │
│ `boolean`, `date`, `datetime` ┆ boolean / date / datetime dtyp… │
│ `categorical` (strings)       ┆ ≤ 500 distinct values           │
│ `free_text` (strings)         ┆ more than 500 distinct values   │
│ `categorical` (numeric)       ┆ ≤ 12 distinct with n ≥ 30, or … │
│ `identifier` (numeric)        ┆ distinct ratio ≥ 0.98           │
│ `continuous` (numeric)        ┆ otherwise                       │
│ `other`                       ┆ dtype the profiler treats as o… │
└───────────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x750966738c80>, _boxhead=Boxhead([ColInfo(var='Role', type=<ColInfoTypeEnum.default: 1>, column_label='Role', column_align='left', column_width=None, stub_level=None), ColInfo(var='Rule', type=<ColInfoTypeEnum.default: 1>, column_label='Rule', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x7509667f5460>, _spanners=Spanners([]), _heading=Heading(title='Role inference', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7509858a6c30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x750965dbfd10>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=6, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=7, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Role', rownum=8, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Rule', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Rule', rownum=1, colnum

Role labels can differ from a column's intended meaning. Numeric
measurements with no repeats, such as random floats, have distinct ratio
1.0 and receive the `identifier` role even when they are not identifiers.
Date and datetime dtypes have their own roles. Similarly, numeric codes
can fall outside the categorical thresholds. Check the label against the
column's meaning. Numeric columns inferred categorical carry
`treat_as_categorical` in the card's `action_hints` list; the finding
that says the same thing (`column_roles`) is a separate, `inferred`-basis
statement.


## Semantic roles

Separately from the card role, the pipeline assigns each column a
semantic role before the detectors run. The rules are applied in
order: a `roles` hint wins; a native date or datetime column is a
`timestamp`; a member of a detected timestamp-part set is a
`timestamp_part` (integer columns named year, month, day, hour,
minute, or second, all values in range, year and month present,
jointly non-null on at least 99% of rows, and the assembled values
valid calendar dates or times on at least 99% of complete rows); an
integer column forming a consecutive run in row order, or restarting
at the same value within every group of a categorical column, is an
`index`; a string or integer column with one distinct value per
non-null row is an `identifier`; a categorical or boolean column is
a `dimension`; a remaining numeric column is a `measure`; a
remaining string column is `text`. Opaque and all-null columns get
no assignment. Each assignment records its evidence — the counts or
the run it rests on — on the card and in the report's `frame`
section, so a wrong assignment can be spotted and overridden with a
hint.

The same pass assembles the frame's time axis: the `time` hint, a
native date or datetime column, a string column that parses as
timestamps across at least 99% of its non-null values, the part set,
or a single year-like integer column recorded at low confidence.
String and part axes are materialized as a real date or datetime
column in the table the temporal passes receive. Index, identifier,
and timestamp columns, and the parts of an assembled axis, are
excluded as detector targets and as discontinuity axes; the
exclusions are recorded in coverage, and the cards still describe
every column.

In [2]:
import polars as pl

import onceover

n = 2_000
df = pl.DataFrame({
    "flag": [True, False] * (n // 2),
    "code": [i % 7 for i in range(n)],          # 7-level numeric code
    "tiny": [i % 2 for i in range(n)],           # 2-level numeric code
    "random_float": [i * 0.5 for i in range(n)],  # distinct ratio 1.0
})
report = onceover.profile(df, budget="quick")
for name in ["flag", "code", "tiny", "random_float"]:
    card = report.columns[name]
    print(f"{name:14s} role={card['role']:12s} distinct={card['distinct']:6d} "
          f"hints={card.get('action_hints', [])}")

flag           role=boolean      distinct=     2 hints=[]
code           role=categorical  distinct=     7 hints=['treat_as_categorical']
tiny           role=categorical  distinct=     2 hints=['treat_as_categorical']
random_float   role=identifier   distinct=  2000 hints=[]


## Concentration

The `concentration` block summarizes the value distribution with three
complementary measures, all exact:

- **HHI** — the Herfindahl–Hirschman index, the sum of squared shares.
  A constant scores 1.0; k equally frequent values score 1/k.
- **Shannon entropy** in nats, and its exponent, **effective
  cardinality** — "this column behaves like it has N distinct values".
  A six-level column dominated by one level scores near 1; a balanced
  six-level column scores near 6.
- **top-1 and top-3 shares** — the mass of the most frequent values.

Concentration findings use fixed thresholds, and each finding states
the number that triggered it: a constant (`distinct ≤ 1`), a
near-constant (top-1 share ≥ 0.99), and a high-concentration flag when the
top value holds ≥ 90% of mass, reported with the effective cardinality.
A column below these thresholds receives no concentration finding.

## Format patterns

String cards carry `format_patterns`: signatures in which digits become
`#`, lowercase letters `a`, uppercase letters `A`, and other characters
are retained. Signatures are truncated at 24 characters. For example,
invoices split between `AAA-####` and `####` produce two patterns, each
with its share. A `formats` finding requires the top pattern to account
for less than 90% of values and the second for at least 5%. The finding
includes both shares and example values.

These signatures can expose mixed date formats, padding, or inconsistent
code formats. They describe character shapes; they do not validate dates,
currencies, phone numbers, or other domain-specific values.
